In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# Install the missing libraries run me on every session's beginning
!pip install -q trl peft bitsandbytes

In [2]:
# Install Weights & Biases quietly
!pip install wandb -qU
import wandb

# This will prompt us for the API key. We just paste it and hit enter.
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ramykhelfaoui (ramykhelfaoui-university-of-boumerdas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# Install the core stack for QLoRA
#!pip install -q -U bitsandbytes
#!pip install -q -U transformers
#!pip install -q -U peft
#!pip install -q -U accelerate
#!pip install -q -U datasets 
# For Multimodal vision support
#!pip install -q -U flash-attn

In [ ]:
# Force-update the hub first to fix the 'KernelInfo' ImportError
!pip install -q -U huggingface_hub
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

In [ ]:
#import torch
#print(f"Is CUDA available? {torch.cuda.is_available()}")
#print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [3]:
#run me on every session's beginning
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
import torch

model_id = "llava-hf/llava-1.5-7b-hf"

# 1. Define the Quantization Configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", # 'nf4' is better for medical/precise reasoning than 'fp4'
    bnb_4bit_use_double_quant=True
)

# 2. Load Processor
processor = AutoProcessor.from_pretrained(model_id)

# 3. Load the Model using the config
model = LlavaForConditionalGeneration.from_pretrained(
    model_id, 
    quantization_config=quant_config, # This replaces the direct 'load_in_4bit'
    device_map="auto",               # Automatically puts model on the T4 GPUs
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("Architecture Defined: LLaVA-1.5-7B loaded correctly with BitsAndBytesConfig.")

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Architecture Defined: LLaVA-1.5-7B loaded correctly with BitsAndBytesConfig.


In [4]:
#run me on every session's beginning
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare the model for 4-bit training (handles gradients/norms)
model = prepare_model_for_kbit_training(model)

# 2. Define the Tuning Parameters
# This ensures we only train a tiny fraction of the model (the 'Adapters')
config = LoraConfig(
    r=64, 
    lora_alpha=128, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # The attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Create the Peft Model
model = get_peft_model(model, config)

# 4. Print the summary to see how many parameters we are training
model.print_trainable_parameters()

trainable params: 76,546,048 || all params: 7,139,973,120 || trainable%: 1.0721


In [ ]:
import os
import json
from PIL import Image
import numpy as np

# 1. Create a dummy image
if not os.path.exists('mock_images'):
    os.makedirs('mock_images')
    # CORRECTED: np.uint8 is the data type, np.random.rand generates the numbers
    random_pixels = np.random.rand(224, 224, 3) * 255
    dummy_img = Image.fromarray(random_pixels.astype('uint8'))
    dummy_img.save('mock_images/test_1.jpg')

# 2. Create a dummy JSONL file
mock_data = [ 
    {"image": "test_1.jpg", "text": "### Human: Analyze this. ### Assistant: This is a mock test."}
]

with open('mock_data.jsonl', 'w') as f:
    for entry in mock_data:
        f.write(json.dumps(entry) + '\n')

print("Mock Data Environment Created successfully.")

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. LOAD THE DATA
dataset = load_dataset("json", data_files="mock_data.jsonl", split="train")

# 2. DEFINE THE TRAINING RULES
training_args = TrainingArguments(
    output_dir="./medical_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,  
    fp16=True,
    logging_steps=1,
    max_steps=5, 
    optim="paged_adamw_32bit",
    remove_unused_columns=False
)

# 3. INITIALIZE THE TRAINER
# We REMOVED peft_config=config because the 'model' variable is already 
# a PeftModel. Passing it again would be "double-dipping."
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("Trainer successfully initialized! The pipeline is 100% verified.")

In [ ]:
import os
# Change 'working' to 'input'
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# Update the path to the JSONL file
DATA_PATH = "/kaggle/input/datasets/ramykhelfaoui/medical-project-proccessed-data/metadata.jsonl"

# Update the path to the images (the 'raw' folder)
IMAGE_FOLDER = "/kaggle/input/datasets/ramykhelfaoui/medical-project-proccessed-data/raw"

In [ ]:
from datasets import load_dataset

# 1. LOAD THE REAL DATA
# We point it to the JSONL file we found in /kaggle/input
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# 2. OPTIONAL: PREVIEW THE DATA
# Let's see the first entry to make sure it looks right
print(f"Dataset loaded! Total examples: {len(dataset)}")
print("First entry sample:")
print(dataset[0])

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# 1. The formatting function
def formatting_prompts_func(example):
    text = f"### Instruction: {example['instruction']}\n### Response: {example['output']}"
    return text

# 2. Configuration for training - BYPASSING THE CRASH
sft_config = SFTConfig(
    output_dir="./medical_llava_final",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=False,             # Turn off trainer-level AMP (let the 4-bit model handle itself)
    bf16=False,             
    max_grad_norm=0.0,      # <--- THIS IS THE MAGIC FIX. It bypasses the crashing line completely.
    logging_steps=5,
    save_total_limit=1,
    optim="paged_adamw_32bit",
    remove_unused_columns=False,
    dataset_text_field="text", 
    report_to="none"
)

# 3. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    formatting_func=formatting_prompts_func
)

print("Crash bypassed. Ready to train!")

In [ ]:
import matplotlib.pyplot as plt

# 1. START THE ENGINES
print("Starting medical training... See you in a few hours!")
trainer.train()

# 2. SAVE THE MODEL (Critical for your project)
trainer.save_model("./medical_llava_final")
print("Model saved successfully!")

# 3. GENERATE THE PROGRESS GRAPH
if hasattr(trainer.state, 'log_history'):
    history = trainer.state.log_history
    steps = [x['step'] for x in history if 'loss' in x]
    loss = [x['loss'] for x in history if 'loss' in x]

    plt.figure(figsize=(10, 6))
    plt.plot(steps, loss, label='Training Loss', color='#e74c3c', linewidth=2)
    plt.title('Medical LLaVA Training Progress')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.show()

In [ ]:
from PIL import Image
import torch
from transformers import TextStreamer

# path: Pointing directly to the images folder in the dataset
test_image_path = "/kaggle/input/medical-project-proccessed-data/images/ROCOv2_2023_train_034603.jpg"

try:
    image = Image.open(test_image_path)
    print(f"Successfully loaded: {test_image_path}")
    
    # 2. Prepare the prompt
    prompt = "### Instruction: Analyze this medical image and describe the clinical observations.\n### Response:"

    # 3. Process and Generate
    inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True)

    print("\n--- MODEL'S MEDICAL ANALYSIS ---")
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=128)

except Exception as e:
    print(f"Error: {e}")

In [ ]:
import os

# The image we are looking for
target_image = "ROCOv2_2023_train_034603.jpg"

# Walk through the directories to find it
found_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if target_image in files:
        found_path = os.path.join(root, target_image)
        break

if found_path:
    print(f"FOUND IT! The real path is: {found_path}")
else:
    print("Could not find that image. Please check the spelling or the JSON file.")

In [ ]:
import os
from PIL import Image
from transformers import TextStreamer

# 1. Automatically find the FIRST image in your dataset
image_extensions = (".jpg", ".jpeg", ".png")
found_test_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(image_extensions):
            found_test_path = os.path.join(root, file)
            break
    if found_test_path: break

if found_test_path:
    print(f"Testing with image found at: {found_test_path}")
    image = Image.open(found_test_path)
    
    # 2. Use the medical prompt style
    prompt = "### Instruction: Analyze this medical image and describe the clinical observations.\n### Response:"

    # 3. Generate the answer
    inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True)

    print("\n--- MODEL'S MEDICAL ANALYSIS ---")
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=128)
    print("\n--------------------------------")
else:
    print("Zero images found in /kaggle/input. Please check if your dataset is actually attached to the notebook.")

In [ ]:
import requests
from PIL import Image
from io import BytesIO
from transformers import TextStreamer

# 1. Load a sample medical image from the web
url = "https://raw.githubusercontent.com/HKUST-KnowComp/LLaVA-Med/main/images/sample_chest_xray.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))

# 2. Prepare the prompt
prompt = "### Instruction: Analyze this medical image and describe the clinical observations.\n### Response:"

# 3. Use your trained model to generate
inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer, skip_prompt=True)

print("\n--- TESTING TRAINED MODEL ON WEB IMAGE ---")
_ = model.generate(**inputs, streamer=streamer, max_new_tokens=128)

In [ ]:
import os

print("--- SEARCHING FOR YOUR IMAGES ---")
found = False
for root, dirs, files in os.walk("/kaggle/input"):
    # Look for any folder that contains a .jpg or .png
    if any(f.lower().endswith(('.jpg', '.png')) for f in files):
        print(f"Found images in: {root}")
        print(f"Sample file: {files[0]}")
        # Store this path for the next step
        REAL_IMAGE_DIR = root 
        SAMPLE_IMAGE_NAME = files[0]
        found = True
        break

if not found:
    print("CRITICAL: No images found. Is the dataset attached in the right-hand sidebar?")

In [ ]:
from PIL import Image
import torch
from transformers import TextStreamer
import os

# 1. PATH SETUP
test_image_path = "/kaggle/input/datasets/ramykhelfaoui/pictotest/test.jpg"

try:
    image = Image.open(test_image_path).convert("RGB")
    print(f"Testing with: {test_image_path}")

    # 2. THE CRITICAL FIX: Adding the <image> token
    # Most LLaVA models need the <image> tag to align the tokens
    prompt = "### Instruction: <image>\nAnalyze this medical image and describe the clinical observations.\n### Response:"

    # 3. GENERATE
    inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
    
    # Using the tokenizer from the processor
    streamer = TextStreamer(processor.tokenizer, skip_prompt=True)

    print("\n--- TRAINED MODEL ANALYSIS ---")
    # We add a bit of 'temperature' to make the medical description more fluid
    _ = model.generate(
        **inputs, 
        streamer=streamer, 
        max_new_tokens=150,
        do_sample=True,
        temperature=0.2
    )
    print("\n------------------------------")

except Exception as e:
    print(f"Error: {e}")

In [19]:
from PIL import Image
import torch
from transformers import TextStreamer
import os

test_image_path = "/kaggle/input/datasets/ramykhelfaoui/pictotest/test.jpg"

try:
    image = Image.open(test_image_path).convert("RGB")
    
    # Simplified prompt to prevent the model from getting stuck in a loop
    prompt = "USER: <image>\nAnalyze this medical image and describe the clinical observations.are there rib fractures present? ASSISTANT:"

    inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
    streamer = TextStreamer(processor.tokenizer, skip_prompt=True)

    print("\n--- FINAL MODEL ANALYSIS ---")
    _ = model.generate(
        **inputs, 
        streamer=streamer, 
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2 # THIS stops the "###" repeating
    )
    print("\n------------------------------")

except Exception as e:
    print(f"Error: {e}")


--- FINAL MODEL ANALYSIS ---
The patient's chest radiograph showed extensive bilateral lung opacities, which were consistent with pneumothoraxes in both lungs. There was no evidence of pleural thickening or rib fracture.</s>

------------------------------


In [6]:
from datasets import load_dataset

# 1. PATH CONFIGURATION 
train_file_path = "/kaggle/input/datasets/ramykhelfaoui/the1000fortraining/train_metadata.jsonl"
eval_file_path = "/kaggle/input/datasets/ramykhelfaoui/the99fortest/test_data.jsonl"

# 2. LOAD paths SEPARATELY
print("Loading datasets into memory separately...")
train_dataset = load_dataset("json", data_files=train_file_path)["train"]
eval_dataset = load_dataset("json", data_files=eval_file_path)["train"] # It defaults to "train" split for single files

# 3. FIX THE TEAMMATE'S DATA MISMATCH
# If the test set has that extra "image" column, we strip it out so the structures match
if "image" in eval_dataset.column_names:
    eval_dataset = eval_dataset.remove_columns("image")

print(f"✅ Loaded {len(train_dataset)} training images.")
print(f"✅ Loaded {len(eval_dataset)} evaluation images.")




Loading datasets into memory separately...
✅ Loaded 1000 training images.
✅ Loaded 100 evaluation images.


In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

# 2. Define the LoRA (Adapter) Configuration
# These are the small layers that will actually be trained
peft_config = LoraConfig(
    r=64, 
    lora_alpha=128, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Targets the attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Wrap the model with these adapters
model = get_peft_model(model, peft_config)

# 4. Verification
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 76,546,048 || all params: 7,139,973,120 || trainable%: 1.0721


In [10]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
import torch
import os

# 1. FORCE CONVERSION AGAIN
model.to(torch.float16)

# 2. CONFIG
output_path = "./medical_llava_final"
def formatting_prompts_func(example):
    return f"### Instruction: {example['instruction']}\n### Response: {example['output']}"

# 3. TRAINING ARGS
training_args = TrainingArguments(
    output_dir=output_path,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    report_to="wandb",
    run_name="medical-llava-no-scaler",
    logging_steps=5,
    
    # HARDWARE
    fp16=False, 
    bf16=False,
    
    # THE SECRET SARE: This stops the GradScaler from checking for BFloat16
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
)

# 4. INITIALIZE
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_prompts_func,
)

# --- THE FIX FOR THE ERROR ---
# We manually disable the scaler in the accelerator to bypass the NotImplementedError
trainer.accelerator.scaler = None 

# 5. RUN
print("🛡️ GradScaler disabled to bypass T4 hardware limitation. Launching...")
trainer.train()

# 6. SAVE
trainer.save_model(output_path)

🛡️ GradScaler disabled to bypass T4 hardware limitation. Launching...


Step,Training Loss
5,2.540071
10,1.897652
15,1.622354
20,1.454792
25,1.360067
30,1.392171
35,1.450729
40,1.424317
45,1.347410
50,1.374633


In [15]:
# Splits the data: 90% for training, 10% for testing
split_data = train_dataset.train_test_split(test_size=0.1)
train_dataset = split_data["train"]
eval_dataset = split_data["test"]

In [16]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
import torch
import os

# --- 1. PRE-FLIGHT HARDWARE CHECK ---
# Force everything to float16 to avoid the BFloat16 crash on T4
model.to(torch.float16)
if hasattr(model, "vision_tower"):
    model.vision_tower.to(dtype=torch.float16)

# --- 2. CONFIG & PROMPT FORMAT ---
output_path = "./medical_llava_perfect"

def formatting_prompts_func(example):
    # Matches the specific format we used in training to prevent ''###'' looping
    return f"### Instruction: {example['instruction']}\n### Response: {example['output']}"

# --- 3. THE PERFECT TRAINING ARGUMENTS ---
training_args = TrainingArguments(
    output_dir=output_path,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=5,               # Higher epochs because Early Stopping will catch it
    
    # Evaluation & Saving (The "Best Version" Logic)
    eval_strategy="steps",            # Check performance every X steps
    eval_steps=20,
    save_strategy="steps",            # Save checkpoints every X steps
    save_steps=20,
    load_best_model_at_end=True,      # CRITICAL: Reverts to the best weights at the end
    metric_for_best_model="loss",     # Uses lowest loss to define "best"
    greater_is_better=False,
    save_total_limit=2,               # Only keep 2 checkpoints to save disk space
    
    # Logging & Monitoring
    report_to="wandb",
    run_name="medical-llava-perfect-run",
    logging_steps=5,
    
    # T4 Hardware Safety
    fp16=False,                       # Disabled because we use the manual bypass below
    bf16=False,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

# --- 4. INITIALIZE TRAINER ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,        # Required for Early Stopping
    formatting_func=formatting_prompts_func,
    # Stop if loss doesn't improve for 2 evaluation checks (40 steps total)
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# --- 5. THE T4 BYPASS (CRITICAL) ---
# This prevents the NotImplementedError by killing the GradScaler
trainer.accelerator.scaler = None 

# --- 6. RUN & SAVE ---
print("🚀 Launching Perfect Run with Early Stopping and Best Model Loading...")
torch.cuda.empty_cache()
trainer.train()

# Save the absolute best version of the model
trainer.save_model(output_path)
print(f"✅ Training complete. Best model saved to {output_path}")

Applying formatting function to train dataset:   0%|          | 0/729 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/729 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/729 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

🚀 Launching Perfect Run with Early Stopping and Best Model Loading...


Step,Training Loss,Validation Loss
20,1.073420,1.059528
40,1.076856,1.058574
60,0.692504,1.119884
80,0.741396,1.138405


✅ Training complete. Best model saved to ./medical_llava_perfect


In [18]:
from PIL import Image
import torch
from transformers import TextStreamer
import os

test_image_path = "/kaggle/input/datasets/ramykhelfaoui/pinguinpictest/Screenshot 2026-03-26 163158.png"

try:
    image = Image.open(test_image_path).convert("RGB")
    
    # Simplified prompt to prevent the model from getting stuck in a loop
    prompt = "USER: <image>\nAnalyze this medical image and describe the clinical observations.are there rib fractures present? ASSISTANT:"

    inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
    streamer = TextStreamer(processor.tokenizer, skip_prompt=True)

    print("\n--- FINAL MODEL ANALYSIS ---")
    _ = model.generate(
        **inputs, 
        streamer=streamer, 
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2 # THIS stops the "###" repeating
    )
    print("\n------------------------------")

except Exception as e:
    print(f"Error: {e}")


--- FINAL MODEL ANALYSIS ---
The penguin mascot of Georgia Tech has a black chest, yellow feet, and white belly.</s>

------------------------------


In [20]:
import shutil
from IPython.display import FileLink

# This compresses your model into a single zip file
shutil.make_archive("medical_llava_weights", 'zip', "./medical_llava_perfect")

# This creates a clickable link to download it to your laptop
FileLink(r'medical_llava_weights.zip')

/kaggle/working/medical_llava_weights.zip